In [12]:
import time
import random
import torch
import torch.multiprocessing as mp

## Simple Dataloader

In [13]:
class Dataset:
    def __init__(self, num_samples: int):
        self.num_samples = num_samples
        self.data = torch.rand(self.num_samples, 1, 2)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx]


In [14]:
class SimpleDataLoader:
    def __init__(self, dataset: Dataset, batch_size: int, shuffle: bool = False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_batches = len(dataset) // batch_size

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        
        if self.shuffle:
            indices = torch.randperm(len(indices)).tolist()
        
        for i in range(self.num_batches):
            start_idx = i * self.batch_size
            end_idx = (i+1) * self.batch_size
            batch_idxs = indices[start_idx:end_idx]
            
            batch = [self.dataset[idx] for idx in batch_idxs]
            yield batch


In [15]:
ds = Dataset(num_samples=8)
dl = SimpleDataLoader(dataset=ds, batch_size=4)

In [16]:
ds[:]

tensor([[[0.9833, 0.1372]],

        [[0.5517, 0.2183]],

        [[0.7939, 0.0626]],

        [[0.5404, 0.2037]],

        [[0.8493, 0.9448]],

        [[0.3025, 0.5977]],

        [[0.7044, 0.7457]],

        [[0.9187, 0.8192]]])

In [17]:
for batch in dl:
    print("---")
    print(batch)
    print("---")

---
[tensor([[0.9833, 0.1372]]), tensor([[0.5517, 0.2183]]), tensor([[0.7939, 0.0626]]), tensor([[0.5404, 0.2037]])]
---
---
[tensor([[0.8493, 0.9448]]), tensor([[0.3025, 0.5977]]), tensor([[0.7044, 0.7457]]), tensor([[0.9187, 0.8192]])]
---


## Multiworker Data loader

In [21]:
import random
import torch
import torch.multiprocessing as mp

In [22]:
class Dataset:
    def __init__(self, num_samples: int):
        self.num_samples = num_samples
        self.data = torch.rand(self.num_samples, 1, 2)
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        return self.data[idx]


In [23]:
def worker_fn(dataset: Dataset, index_queue: mp.Queue, data_queue: mp.Queue):
    while True:
        try:
            batch_idxs = index_queue.get()
        except Exception:
            break
        batch = [dataset[idx] for idx in batch_idxs]
        data_queue.put(batch)  # This is a blocking operation, new element will not stack up until queue has space


class MultiWorkerDataLoader:
    def __init__(self, dataset: Dataset, batch_size: int, num_workers: int, shuffle: bool = False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.shuffle = shuffle
        self.num_batches = len(dataset) // batch_size

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        
        if self.shuffle:
            indices = torch.randperm(len(indices)).tolist()
        
        ## Create two queues, one for indices and one for the actual data
        index_queue = mp.Queue(self.num_batches)
        data_queue = mp.Queue(self.num_workers)

        ## Loop through the number of batches like before
        for i in range(self.num_batches):
            start_idx = i * self.batch_size
            end_idx = (i+1) * self.batch_size
            batch_idxs = indices[start_idx:end_idx]
            
            ## Intstead of yielding the batch, put the indices in a queue
            index_queue.put(batch_idxs)
        
        ## Process the indices in a parallel manner
        workers = [
            mp.Process(target=worker_fn, args=(self.dataset, index_queue, data_queue))
            for _ in range(self.num_workers)
        ]
        for worker in workers:
            worker.start()
         
        ## At this point, the workers have processed the batches and just need to come off the queue
        for i in range(self.num_batches):
            batch = data_queue.get()
            yield batch
        
        ## Cleanup
        for worker in workers:
            worker.terminate()
        index_queue.close()
        data_queue.close()


In [24]:
ds = Dataset(num_samples=8)
dl = MultiWorkerDataLoader(dataset=ds, batch_size=2, num_workers=2)

In [25]:
ds[:]

tensor([[[0.7795, 0.6530]],

        [[0.7363, 0.0082]],

        [[0.3901, 0.5586]],

        [[0.7557, 0.8361]],

        [[0.4553, 0.1689]],

        [[0.2950, 0.5642]],

        [[0.6766, 0.2969]],

        [[0.8296, 0.0834]]])

In [26]:
for batch in dl:
    print("---")
    print(batch)
    print("---")

---
[tensor([[0.7795, 0.6530]]), tensor([[0.7363, 0.0082]])]
---
---
[tensor([[0.3901, 0.5586]]), tensor([[0.7557, 0.8361]])]
---
---
[tensor([[0.4553, 0.1689]]), tensor([[0.2950, 0.5642]])]
---
---
[tensor([[0.6766, 0.2969]]), tensor([[0.8296, 0.0834]])]
---


## Latency diff between both data loaders

In [36]:
ds = Dataset(num_samples=320000)

In [37]:
dl = SimpleDataLoader(dataset=ds, batch_size=32)
start_time = time.time()
for batch in dl:
    pass
end_time = time.time()
print(f"Elapsed time = {end_time - start_time}s")

Elapsed time = 0.37101316452026367s


In [38]:
dl = MultiWorkerDataLoader(dataset=ds, batch_size=32, num_workers=2)
start_time = time.time()
for batch in dl:
    pass
end_time = time.time()
print(f"Elapsed time = {end_time - start_time}s")

Elapsed time = 7.336867094039917s


## Add PreFetch
The only diff is that you let the size of the buffer (data_queue) be larger which is `num_workers * prefetch_factor`

In [39]:
def worker_fn(dataset: Dataset, index_queue: mp.Queue, data_queue: mp.Queue):
    while True:
        try:
            batch_idxs = index_queue.get()
        except Exception:
            break
        batch = [dataset[idx] for idx in batch_idxs]
        data_queue.put(batch)  # This is a blocking operation, new element will not stack up until queue has space


class MultiWorkerPreFetchDataLoader:
    def __init__(self, dataset: Dataset, batch_size: int, num_workers: int = 2, prefetch_factor: int = 2, shuffle: bool = False):
        self.dataset = dataset
        self.batch_size = batch_size
        self.num_workers = num_workers
        self.prefetch_factor = prefetch_factor
        self.shuffle = shuffle
        self.num_batches = len(dataset) // batch_size

    def __iter__(self):
        indices = list(range(len(self.dataset)))
        
        if self.shuffle:
            indices = torch.randperm(len(indices)).tolist()
        
        ## Create two queues, one for indices and one for the actual data
        index_queue = mp.Queue(self.num_batches)
        data_queue = mp.Queue(self.num_workers * self.prefetch_factor)

        ## Loop through the number of batches like before
        for i in range(self.num_batches):
            start_idx = i * self.batch_size
            end_idx = (i+1) * self.batch_size
            batch_idxs = indices[start_idx:end_idx]
            
            ## Intstead of yielding the batch, put the indices in a queue
            index_queue.put(batch_idxs)
        
        ## Process the indices in a parallel manner
        workers = [
            mp.Process(target=worker_fn, args=(self.dataset, index_queue, data_queue))
            for _ in range(self.num_workers)
        ]
        for worker in workers:
            worker.start()
         
        ## At this point, the workers have processed the batches and just need to come off the queue
        for i in range(self.num_batches):
            batch = data_queue.get()
            yield batch
        
        ## Cleanup
        for worker in workers:
            worker.terminate()
        index_queue.close()
        data_queue.close()


## Latency diff b/w all 3

In [40]:
ds = Dataset(num_samples=320000)

In [41]:
dl = SimpleDataLoader(dataset=ds, batch_size=32)
start_time = time.time()
for batch in dl:
    pass
end_time = time.time()
print(f"Elapsed time = {end_time - start_time}s")

Elapsed time = 0.3740043640136719s


In [42]:
dl = MultiWorkerDataLoader(dataset=ds, batch_size=32, num_workers=2)
start_time = time.time()
for batch in dl:
    pass
end_time = time.time()
print(f"Elapsed time = {end_time - start_time}s")

Elapsed time = 7.18854546546936s


In [45]:
dl = MultiWorkerPreFetchDataLoader(dataset=ds, batch_size=32, num_workers=2, prefetch_factor=2)
start_time = time.time()
for batch in dl:
    pass
end_time = time.time()
print(f"Elapsed time = {end_time - start_time}s")

Elapsed time = 7.031349420547485s
